<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/VenusRXN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. 拉取核心代码库并进入目录（如果在左侧文件栏已经看到VenusRXN文件夹，可跳过git clone）
!git clone https://github.com/zy-zhou/VenusRXN.git
%cd VenusRXN

# 2. 获取当前Colab底层的PyTorch版本
import torch
pt_version = torch.__version__
print(f"当前检测到的PyTorch版本为: {pt_version}，正在匹配图算子...")

# 3. 安装图网络核心库，利用动态版本号精准定位预编译包以避免源码编译错误
!pip install --upgrade pip
!pip install torch-geometric
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{pt_version}.html

# 4. 手动补齐由于缺少requirements.txt而可能漏掉的常见生化和语言模型依赖
!pip install rdkit transformers fair-esm

Cloning into 'VenusRXN'...
remote: Enumerating objects: 93, done.
remote: Total 93 (delta 0), reused 0 (delta 0), pack-reused 93 (from 1)
Receiving objects: 100% (93/93), 94.13 MiB | 40.76 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/VenusRXN/VenusRXN
当前检测到的PyTorch版本为: 2.10.0+cu128，正在匹配图算子...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.1 MB/s  0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 158.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 135.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 118.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [torch-cluster]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 58.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rdkit]


In [5]:
# 1. 安装专用的 Zenodo 批量下载工具
!pip install zenodo-get

# 2. 创建 checkpoints 文件夹并进入该目录
import os
os.makedirs('checkpoints', exist_ok=True)
%cd checkpoints

# 3. 直接通过 Zenodo 记录号 (13897920) 自动解析真实直链并下载所有文件
!zenodo_get 13897920

# 4. 查看下载下来的文件列表
!ls -l

# 解压当前目录下所有下载好的.zip 文件（加引号防止 shell 提前转义）
!unzip -o "*.zip"

# 正确返回上一级项目主目录（注意 cd 和.. 之间的空格，或者直接使用绝对路径）
%cd /content/VenusRXN

/content/VenusRXN/VenusRXN/checkpoints/checkpoints
INFO: Output directory: /content/VenusRXN/VenusRXN/checkpoints/checkpoints
INFO: Title: ProtNote: a multimodal method for protein-function annotation
INFO: Total size: 77.7 GB
INFO: Number of files: 3
ERROR: Immediate abort. There might be unfinished files.
total 20493488
-rw-r--r-- 1 root root  2267865088 May 10 17:00 ablation_models.zip
-rw-r--r-- 1 root root 18717458173 May 10 16:59 outputs.zip
Archive:  ablation_models.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
Archive:  outputs.zip
   creating: outputs/
   creating: outputs/results/
  inflating: outputs/results/test_1_labels_TEST_DATA_PATH_ZERO_SHOT_LEAF_NODES_seed_replicates_v9_12_sum_last_epoch.h5  
  inflating: outputs/results/test_1_labels_TEST_DATA_PATH_seed_repli

In [7]:
# 1. 强制回到上一次成功下载权重的目录
%cd /content/VenusRXN/VenusRXN/checkpoints

# 2. 解压这里已经下载好的文件
!unzip -o "*.zip"

# 3. 退回正确的工作主目录，准备后续操作
%cd /content/VenusRXN/VenusRXN

OSError: [Errno 28] No space left on device

In [6]:
# 1. 进入 checkpoints 文件夹
%cd /content/VenusRXN/VenusRXN/checkpoints

# 2. 删除巨大且推理时完全不需要的消融实验包及其中断解压的残留文件夹
!rm -rf ablation_models.zip ablation_models/

# 3. 逐个解压核心的权重文件。使用 && 确保解压成功后立即删除原.zip 文件，极限节省空间
!unzip -o data.zip && rm data.zip
!unzip -o outputs.zip && rm outputs.zip

# Move the 'data' directory to the project root as expected by enzyme_retrieval.py
!mv data /content/VenusRXN/

# 4. 解压完毕后，返回工作主目录
%cd /content/VenusRXN/VenusRXN

/content/VenusRXN/VenusRXN/checkpoints
unzip:  cannot find or open data.zip, data.zip.zip or data.zip.ZIP.
unzip:  cannot find or open outputs.zip, outputs.zip.zip or outputs.zip.ZIP.
mv: cannot move 'data' to '/content/VenusRXN/data': Directory not empty
/content/VenusRXN/VenusRXN


In [52]:
import sys
import os
import torch
import re

# Ensure current working directory is the base of the cloned repo
%cd /content/VenusRXN

file_path = '/content/VenusRXN/rxnzyme/models/esm.py'
try:
    with open(file_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    commenting_docstring_block = False

    for line in lines:
        stripped_line = line.strip()

        # If we are currently in a docstring decorator block, handle it first
        if commenting_docstring_block:
            # If the current line is a class or def, stop commenting this block
            # This check applies to stripped_line to ignore leading whitespace
            if stripped_line.startswith('class ') or stripped_line.startswith('def '):
                commenting_docstring_block = False # End the commenting block
                patched_lines.append(line) # Append the class/def line as is
            else:
                # Continue commenting lines within the decorator arguments
                patched_lines.append('# ' + line)
        # If not currently commenting a docstring block, check for the start of one
        # Use \s* to account for potential leading whitespace before the decorator
        elif re.search(r"^\s*@add_start_docstrings_to_model_forward", line) or \
             re.search(r"^\s*@add_code_sample_docstrings", line):
            patched_lines.append('# ' + line) # Comment the decorator line
            commenting_docstring_block = True # Start the commenting block
        # Handle transformers.modeling_utils import
        elif re.search(r"from\s+transformers\.modeling_utils\s+import.*find_pruneable_heads_and_indices", line):
            patched_lines.append('from transformers.modeling_utils import PreTrainedModel # Modified by Colab Agent to fix ImportError\n')
        # Handle transformers.file_utils import
        elif re.search(r"from\s+transformers\.file_utils\s+import", line):
            patched_lines.append('# ' + line)
        else:
            patched_lines.append(line)

    # Write the patched content back to the file
    with open(file_path, 'w') as f:
        f.writelines(patched_lines)

    print("rxnzyme/models/esm.py has been patched to comment out problematic decorators and adjust imports.")

    # Verification step: Read the file again and print relevant lines
    with open(file_path, 'r') as f_verify:
        verify_lines = f_verify.readlines()
    print("\n--- Verification of patched lines (around line 755) ---")
    start_line_display = max(0, 755 - 1 - 5)
    end_line_display = min(len(verify_lines), 755 - 1 + 5)
    for i in range(start_line_display, end_line_display):
        print(f"{i+1}: {verify_lines[i].strip()}")
    print("-----------------------------------------------------")

except Exception as e:
    print(f"Error during patching: {e}")

/content/VenusRXN
rxnzyme/models/esm.py has been patched to comment out problematic decorators and adjust imports.

--- Verification of patched lines (around line 755) ---
750: class PreTrainedModel
751: """
752: for layer, heads in heads_to_prune.items():
753: self.encoder.layer[layer].attention.prune_heads(heads)
754: 
755: #     @add_start_docstrings_to_model_forward(ESM_INPUTS_DOCSTRING.format("(batch_size, sequence_length)"))
756: #     @add_code_sample_docstrings(
757: #         checkpoint=_CHECKPOINT_FOR_DOC,
758: #         output_type=BaseModelOutputWithPoolingAndCrossAttentions,
759: #         config_class=_CONFIG_FOR_DOC,
-----------------------------------------------------


In [14]:
import sys
import os
import torch

# Add the project root to sys.path for module discovery
project_root = '/content/VenusRXN'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch rxnzyme/training/base.py
base_py_path = os.path.join(project_root, 'rxnzyme', 'training', 'base.py')
try:
    with open(base_py_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    for line in lines:
        if "from lightning import LightningModule, Trainer" in line:
            patched_lines.append("from pytorch_lightning import LightningModule, Trainer\n")
        elif "from lightning.pytorch.loggers import CSVLogger" in line:
            patched_lines.append("from pytorch_lightning.loggers import CSVLogger\n")
        elif "from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping" in line:
            patched_lines.append("from pytorch_lightning import ModelCheckpoint, EarlyStopping\n")
        else:
            patched_lines.append(line)

    with open(base_py_path, 'w') as f:
        f.writelines(patched_lines)
    print(f"Patched {base_py_path} to use `from pytorch_lightning import ...`")
except Exception as e:
    print(f"Error patching {base_py_path}: {e}")

# Patch rxnzyme/extractor.py
extractor_py_path = os.path.join(project_root, 'rxnzyme', 'extractor.py')
try:
    with open(extractor_py_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    for line in lines:
        if "from lightning import LightningModule" in line:
            patched_lines.append("from pytorch_lightning import LightningModule\n")
        else:
            patched_lines.append(line)

    with open(extractor_py_path, 'w') as f:
        f.writelines(patched_lines)
    print(f"Patched {extractor_py_path} to use `from pytorch_lightning import ...`")
except Exception as e:
    print(f"Error patching {extractor_py_path}: {e}")

# Patch rxnzyme/data_modules/prorxn.py
prorxn_data_module_path = os.path.join(project_root, 'rxnzyme', 'data_modules', 'prorxn.py')
try:
    with open(prorxn_data_module_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    for line in lines:
        if "from lightning import LightningDataModule" in line:
            patched_lines.append("from pytorch_lightning import LightningDataModule\n")
        else:
            patched_lines.append(line)

    with open(prorxn_data_module_path, 'w') as f:
        f.writelines(patched_lines)
    print(f"Patched {prorxn_data_module_path} to use `from pytorch_lightning import ...`")
except Exception as e:
    print(f"Error patching {prorxn_data_module_path}: {e}")

# Patch rxnzyme/data_modules/extractor.py
extractor_data_module_path = os.path.join(project_root, 'rxnzyme', 'data_modules', 'extractor.py')
try:
    with open(extractor_data_module_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    for line in lines:
        if "from lightning import LightningDataModule" in line:
            patched_lines.append("from pytorch_lightning import LightningDataModule\n")
        else:
            patched_lines.append(line)

    with open(extractor_data_module_path, 'w') as f:
        f.writelines(patched_lines)
    print(f"Patched {extractor_data_module_path} to use `from pytorch_lightning import ...`")
except Exception as e:
    print(f"Error patching {extractor_data_module_path}: {e}")


!pip install pytorch-lightning

# 1. Comment out the incorrect import paths for ReactionEncoder and JointEncoder.
#    These classes are not directly exportable from these modules.
# from rxnzyme.extractor import ReactionEncoder, JointEncoder
# from enzyme_retrieval import ReactionEncoder, JointEncoder

# Instead, these encoders are likely components of the 'prorxn' model.
# We will instantiate the prorxn model and then inspect its attributes to find the encoders.

# The cat command below is only for inspection, not for execution.
# !cat /content/VenusRXN/rxnzyme/extractor.py

# 2. Load pre-generated ESM-2 650M feature embeddings
# NOTE: Please verify the filename 'nocardioides_esm2_650M_embeddings.npy'.
# You can right-click the .pt file in the left file pane and select 'Copy path' to ensure accuracy.
# Path is relative to the project root /content/VenusRXN
# feature_path = '/content/VenusRXN/data/nocardioides_esm2_650M_embeddings.npy'
# print(f"正在载入本地全基因组特征矩阵: {feature_path}...")
# genome_embeddings_db = torch.load(feature_path)

# 3. Mount large model weights (use absolute paths, pointing to your unzipped data/models directory)
# print("正在将大模型权重加载至 A100 GPU...")
# Paths are relative to the project root /content/VenusRXN
# reaction_encoder_path = '/content/VenusRXN/data/models/reaction_weights'
# jin_encoder_path = '/content/VenusRXN/data/models/joint_weights'

# reaction_encoder = ReactionEncoder.from_pretrained(reaction_encoder_path).cuda()
# joint_encoder = JointEncoder.from_pretrained(jin_encoder_path).cuda()
# reaction_encoder.eval()
# joint_encoder.eval()

# 4. Strictly construct the reaction SMILES for DON to 3-epi-DON (including precise chiral markers)
# don_smiles = "CC1=C[C@@H]2[C@]([C@@H](C1=O)O)([C@]3(C[C@H]([C@H]([C@@]34CO4)O)O)C)CO"
# epi_don_smiles = "OC[C@@]12[C@@H](C=C(C(=O)[C@H]1O)C)O[C@H]1[C@]3([C@]2(C)C[C@H]1O)CO3"
# query_reaction_smiles = f"{don_smiles}>>{epi_don_smiles}"

# 5. Start deep learning multi-modal retrieval
# print("启动多模态图网络，正在计算反应拓扑变异并执行特征空间比对...")
# top_candidates = joint_encoder.retrieve_enzymes(
#     query_reaction=query_reaction_smiles,
#     protein_database=genome_embeddings_db,
#     top_k=500 # Extract top 500 candidate sequences
# )

# print(f"Retrieved top_candidates: {top_candidates}") # Debugging print

# if top_candidates is None:
#     print("Warning: joint_encoder.retrieve_enzymes returned None. No candidates found or an error occurred.")
#     # Optionally, you can add more specific error handling or exit here.
# else:
#     print("\n针对 DON 生成 3-epi-DON 的首选潜在催化靶点列表 (Top 500)：")
#     results = []
#     for rank, candidate in enumerate(top_candidates):
#         print(f"Rank {rank+1}: 基因ID {candidate['id']}, 跨模态评分: {candidate['score']:.4f}")
#         results.append(f"{candidate['id']},{candidate['score']:.4f}")

#     # 6. Persist results to CSV (saved in the main directory for easy download)
#     # Path is relative to the project root /content/VenusRXN
#     output_csv = "/content/VenusRXN/top_500_candidates.csv"
#     with open(output_csv, "w") as f:
#         f.write("Rank,Gene_ID,Score\n")
#         for i, res in enumerate(results):
#             f.write(f"{i+1},{res}\n")

#     print(f"\n推断彻底完成！数据已保存至 {output_csv}，可从左侧文件树下载。")

Patched /content/VenusRXN/rxnzyme/training/base.py to use `from pytorch_lightning import ...`
Patched /content/VenusRXN/rxnzyme/extractor.py to use `from pytorch_lightning import ...`
Patched /content/VenusRXN/rxnzyme/data_modules/prorxn.py to use `from pytorch_lightning import ...`
Patched /content/VenusRXN/rxnzyme/data_modules/extractor.py to use `from pytorch_lightning import ...`


In [77]:
!python enzyme_retrieval.py --ref_enzymes

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
Traceback (most recent call last):
  File "/content/VenusRXN/enzyme_retrieval.py", line 6, in <module>
    from rxnzyme.models.loading import get_tokenizer, get_plm, get_prorxn
  File "/content/VenusRXN/rxnzyme/models/loading.py", line 3, in <module>
    from transformers import EsmConfig, EsmTokenizer
ModuleNotFoundError: No module named 'transformers'


In [78]:
# Install transformers library to resolve ModuleNotFoundError
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 28.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.3 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
esm 3.2.3 requires transformers<4.48.2, but you have transformers 5.8.0 which is incompatible.


In [79]:
# Re-run the patching cell for rxnzyme/models/esm.py to ensure compatibility after reinstalling transformers
# This cell was originally '7MZj4EF7-bnk'
import sys
import os
import torch
import re

# Ensure current working directory is the base of the cloned repo
%cd /content/VenusRXN

file_path = '/content/VenusRXN/rxnzyme/models/esm.py'
try:
    with open(file_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    commenting_docstring_block = False

    for line in lines:
        stripped_line = line.strip()

        # If we are currently in a docstring decorator block, handle it first
        if commenting_docstring_block:
            # If the current line is a class or def, stop commenting this block
            # This check applies to stripped_line to ignore leading whitespace
            if stripped_line.startswith('class ') or stripped_line.startswith('def '):
                commenting_docstring_block = False # End the commenting block
                patched_lines.append(line) # Append the class/def line as is
            else:
                # Continue commenting lines within the decorator arguments
                patched_lines.append('# ' + line)
        # If not currently commenting a docstring block, check for the start of one
        # Use \s* to account for potential leading whitespace before the decorator
        elif re.search(r"^\s*@add_start_docstrings_to_model_forward", line) or \
             re.search(r"^\s*@add_code_sample_docstrings", line):
            patched_lines.append('# ' + line) # Comment the decorator line
            commenting_docstring_block = True # Start the commenting block
        # Handle transformers.modeling_utils import
        elif re.search(r"from\s+transformers\\.modeling_utils\s+import.*find_pruneable_heads_and_indices", line):
            patched_lines.append('from transformers.modeling_utils import PreTrainedModel # Modified by Colab Agent to fix ImportError\n')
        # Handle transformers.file_utils import
        elif re.search(r"from\s+transformers\\.file_utils\s+import", line):
            patched_lines.append('# ' + line)
        else:
            patched_lines.append(line)

    # Write the patched content back to the file
    with open(file_path, 'w') as f:
        f.writelines(patched_lines)

    print("rxnzyme/models/esm.py has been patched to comment out problematic decorators and adjust imports.")

    # Verification step: Read the file again and print relevant lines
    with open(file_path, 'r') as f_verify:
        verify_lines = f_verify.readlines()
    print("\n--- Verification of patched lines (around line 755) ---")
    start_line_display = max(0, 755 - 1 - 5)
    end_line_display = min(len(verify_lines), 755 - 1 + 5)
    for i in range(start_line_display, end_line_display):
        print(f"{i+1}: {verify_lines[i].strip()}")
    print("-----------------------------------------------------")

except Exception as e:
    print(f"Error during patching: {e}")

/content/VenusRXN
rxnzyme/models/esm.py has been patched to comment out problematic decorators and adjust imports.

--- Verification of patched lines (around line 755) ---
750: class PreTrainedModel
751: """
752: for layer, heads in heads_to_prune.items():
753: self.encoder.layer[layer].attention.prune_heads(heads)
754: 
755: #     @add_start_docstrings_to_model_forward(ESM_INPUTS_DOCSTRING.format("(batch_size, sequence_length)"))
756: #     @add_code_sample_docstrings(
757: #         checkpoint=_CHECKPOINT_FOR_DOC,
758: #         output_type=BaseModelOutputWithPoolingAndCrossAttentions,
759: #         config_class=_CONFIG_FOR_DOC,
-----------------------------------------------------


In [4]:
!python enzyme_retrieval.py --ref_enzymes

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
2026-05-10 18:24:27.731234: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-10 18:24:28.945026: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trac

In [1]:
# Uninstall current transformers, huggingface-hub, and tokenizers to resolve version conflicts
!pip uninstall -y transformers huggingface-hub tokenizers

Found existing installation: transformers 5.8.0
Uninstalling transformers-5.8.0:
  Successfully uninstalled transformers-5.8.0
Found existing installation: huggingface_hub 1.14.0
Uninstalling huggingface_hub-1.14.0:
  Successfully uninstalled huggingface_hub-1.14.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2


In [8]:
# Install compatible versions of transformers and tokenizers
!pip install tokenizers==0.15.0
!pip uninstall -y transformers
!pip install transformers==4.47.0

  Using cached tokenizers-0.15.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached tokenizers-0.15.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.8 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.47.0 requires tokenizers<0.22,>=0.21, but you have tokenizers 0.15.0 which is incompatible.
Found existing installation: transformers 4.47.0
Uninstalling transformers-4.47.0:
  Successfully uninstalled transformers-4.47.0
  Using cached transformers-4.47.0-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.47.

In [9]:
# Reinstall transformers==4.47.0 after tokenizers has been installed
!pip install transformers==4.47.0

In [10]:
# Re-run the patching cell for rxnzyme/models/esm.py to ensure compatibility after reinstalling transformers
# This cell was originally '7MZj4EF7-bnk'
import sys
import os
import torch
import re

# Ensure current working directory is the base of the cloned repo
%cd /content/VenusRXN

file_path = '/content/VenusRXN/rxnzyme/models/esm.py'
try:
    with open(file_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    commenting_docstring_block = False

    for line in lines:
        stripped_line = line.strip()

        # If we are currently in a docstring decorator block, handle it first
        if commenting_docstring_block:
            # If the current line is a class or def, stop commenting this block
            # This check applies to stripped_line to ignore leading whitespace
            if stripped_line.startswith('class ') or stripped_line.startswith('def '):
                commenting_docstring_block = False # End the commenting block
                patched_lines.append(line) # Append the class/def line as is
            else:
                # Continue commenting lines within the decorator arguments
                patched_lines.append('# ' + line)
        # If not currently commenting a docstring block, check for the start of one
        # Use \s* to account for potential leading whitespace before the decorator
        elif re.search(r"^\s*@add_start_docstrings_to_model_forward", line) or \
             re.search(r"^\s*@add_code_sample_docstrings", line):
            patched_lines.append('# ' + line) # Comment the decorator line
            commenting_docstring_block = True # Start the commenting block
        # Handle transformers.modeling_utils import
        elif re.search(r"from\s+transformers\\.modeling_utils\s+import.*find_pruneable_heads_and_indices", line):
            patched_lines.append('from transformers.modeling_utils import PreTrainedModel # Modified by Colab Agent to fix ImportError\n')
        # Handle transformers.file_utils import
        elif re.search(r"from\s+transformers\\.file_utils\s+import", line):
            patched_lines.append('# ' + line)
        else:
            patched_lines.append(line)

    # Write the patched content back to the file
    with open(file_path, 'w') as f:
        f.writelines(patched_lines)

    print("rxnzyme/models/esm.py has been patched to comment out problematic decorators and adjust imports.")

    # Verification step: Read the file again and print relevant lines
    with open(file_path, 'r') as f_verify:
        verify_lines = f_verify.readlines()
    print("\n--- Verification of patched lines (around line 755) ---")
    start_line_display = max(0, 755 - 1 - 5)
    end_line_display = min(len(verify_lines), 755 - 1 + 5)
    for i in range(start_line_display, end_line_display):
        print(f"{i+1}: {verify_lines[i].strip()}")
    print("-----------------------------------------------------")

except Exception as e:
    print(f"Error during patching: {e}")

/content/VenusRXN
rxnzyme/models/esm.py has been patched to comment out problematic decorators and adjust imports.

--- Verification of patched lines (around line 755) ---
750: class PreTrainedModel
751: """
752: for layer, heads in heads_to_prune.items():
753: self.encoder.layer[layer].attention.prune_heads(heads)
754: 
755: #     @add_start_docstrings_to_model_forward(ESM_INPUTS_DOCSTRING.format("(batch_size, sequence_length)"))
756: #     @add_code_sample_docstrings(
757: #         checkpoint=_CHECKPOINT_FOR_DOC,
758: #         output_type=BaseModelOutputWithPoolingAndCrossAttentions,
759: #         config_class=_CONFIG_FOR_DOC,
-----------------------------------------------------


In [11]:
# Re-run the enzyme retrieval script now that dependencies and patches are in place, and data is assumed to be ready.
!python enzyme_retrieval.py --ref_enzymes

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
2026-05-10 18:29:02.356285: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-10 18:29:02.438811: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trac

In [15]:
# 1. Navigate to the checkpoints directory
%cd /content/VenusRXN/checkpoints

# 2. Clean up any previous partial downloads or extractions of 'data'
!rm -rf data.zip data

# 3. Explicitly download the missing 'data.zip' file from Zenodo (it's ~56GB)
#    This step might take a long time and requires sufficient disk space.
!wget -O data.zip "https://zenodo.org/records/13897920/files/data.zip?download=1"

# 4. Unzip the 'data.zip' file. Use -o to overwrite existing files if any.
!unzip -o data.zip

# 5. Remove the large zip file after extraction to save space.
!rm data.zip

# 6. Move the extracted 'data' directory to the project root as expected by enzyme_retrieval.py
#    The 'data' directory will be created inside the current directory first, so we move it up.
!mv data /content/VenusRXN/

# 7. Return to the main project directory
%cd /content/VenusRXN

# 8. Re-run the enzyme retrieval script now that the data is in place
!python enzyme_retrieval.py --ref_enzymes

/content/VenusRXN/checkpoints
--2026-05-10 18:38:01--  https://zenodo.org/records/13897920/files/data.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.103.118, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17594525298 (16G) [application/octet-stream]
Saving to: ‘data.zip’

data.zip              3%[                    ] 540.86M   107MB/s    eta 2m 37s ^C
Archive:  data.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of data.zip or
        data.zip.zip, and cannot find data.zip.ZIP, period.
mv: cannot stat 'data': No such file or directory
/content/VenusRXN
object address  : 0x78e64ee902e0
object refcount : 3


In [73]:
import os

# Inspect the content of enzyme_retrieval.py to find ReactionEncoder and JointEncoder
retrieval_file_path = '/content/VenusRXN/enzyme_retrieval.py'
if os.path.exists(retrieval_file_path):
    print(f"Content of {retrieval_file_path}:")
    with open(retrieval_file_path, 'r') as f:
        print(f.read())
else:
    print(f"File not found: {retrieval_file_path}")

Content of /content/VenusRXN/enzyme_retrieval.py:
import os
import argparse
import torch
import pandas as pd
from rxnzyme.data.datasets import ignore_label
from rxnzyme.models.loading import get_tokenizer, get_plm, get_prorxn
from rxnzyme.training.base import get_trainer
from rxnzyme.training.prorxn import LitProRxnForMM
from rxnzyme.extractor import plm_mean_pooling, DenseRetriever
from rxnzyme.data_modules.prorxn import ProRxnDataModule
from rxnzyme.data_modules.extractor import DenseRetrieverDataModule
from rxnzyme.utils import read_json, read_fasta, retrieval_metrics, screening_metrics

rxn_graphormer_config = read_json('configs/rxn_graphormer.json')
mol_graphormer_config = rxn_graphormer_config['mol_graphormer']
cgr_graphormer_config = rxn_graphormer_config['cgr_graphormer']
train_config = read_json('configs/prorxn_pretrain.json')

def get_pair_ids(rxn_db_dir, enz_db_path, ids_path):
    pair_ids = pd.read_csv(
        ids_path,
        sep='\t' if ids_path.endswith('.tsv') else '

In [45]:
import os

file_path = '/content/VenusRXN/rxnzyme/models/esm.py'
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    # Display lines around the reported error (line 755, adjust for 0-indexed list)
    start_line = max(0, 755 - 1 - 5) # 5 lines before
    end_line = min(len(lines), 755 - 1 + 5) # 5 lines after
    print(f"Content of {file_path} (lines {start_line+1}-{end_line+1}):\n")
    for i in range(start_line, end_line):
        print(f"{i+1}: {lines[i].strip() if i < len(lines) else ''}")
else:
    print(f"File not found: {file_path}")


Content of /content/VenusRXN/rxnzyme/models/esm.py (lines 750-760):

750: class PreTrainedModel
751: """
752: for layer, heads in heads_to_prune.items():
753: self.encoder.layer[layer].attention.prune_heads(heads)
754: 
755: @add_start_docstrings_to_model_forward(ESM_INPUTS_DOCSTRING.format("(batch_size, sequence_length)"))
756: @add_code_sample_docstrings(
757: checkpoint=_CHECKPOINT_FOR_DOC,
758: output_type=BaseModelOutputWithPoolingAndCrossAttentions,
759: config_class=_CONFIG_FOR_DOC,


In [18]:
# Install the missing lmdb package
!pip install lmdb

In [20]:
# Install the Biopython library
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.1 MB/s  0:00:00


In [24]:
# Install the ESM library
!pip install esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 92.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 139.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 47.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 197.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 131.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.3 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing inst

In [26]:
# The 'transformers' library installed by 'esm' (4.48.1) is causing an ImportError due to deprecated modules.
# Uninstall it to replace with a compatible version.
!pip uninstall -y transformers

Found existing installation: transformers 4.48.1
Uninstalling transformers-4.48.1:
  Successfully uninstalled transformers-4.48.1


In [27]:
# Install a compatible version of 'transformers' (e.g., 4.20.0) that still includes 'file_utils'.
!pip install transformers==4.20.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 10.0 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [30]:
# Explicitly install a compatible version of tokenizers with pre-built wheels
!pip install tokenizers==0.12.1

  Using cached tokenizers-0.12.1.tar.gz (220 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [31]:
# Re-install transformers==4.20.0, which should now find the pre-installed tokenizers
!pip install transformers==4.20.0

  Using cached transformers-4.20.0-py3-none-any.whl.metadata (77 kB)
  Using cached tokenizers-0.12.1.tar.gz (220 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... canceledERROR: Operation cancelled by user


In [28]:
# Explicitly install a compatible version of tokenizers with pre-built wheels
!pip install tokenizers==0.12.1

  Using cached tokenizers-0.12.1.tar.gz (220 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [29]:
# Re-install transformers==4.20.0, which should now find the pre-installed tokenizers
!pip install transformers==4.20.0

  Using cached transformers-4.20.0-py3-none-any.whl.metadata (77 kB)
  Using cached tokenizers-0.12.1.tar.gz (220 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.20.0-py3-none-any.whl (4.4 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
